# 11 — Razonamiento con Incertidumbre: ANFIS para Predicción de ETA

Sistema de Inferencia Neuro-Difuso Adaptativo (ANFIS) — Entrega 3.

ANFIS combina redes neuronales (para aprender los parámetros) con lógica difusa (para razonar bajo incertidumbre).
Predice `dias_entrega` usando 3 variables lingüísticas con grados de pertenencia interpretables en lenguaje natural.

## Arquitectura del Sistema Difuso (Sugeno tipo-1, Jang 1993)

Las 5 capas funcionales del ANFIS:

| Capa | Función | Parámetros |
|------|---------|-----------|
| L1 Fuzzificación | µᵢⱼ(xᵢ) — grado de pertenencia de xᵢ al conjunto j | Premisa (a,b,c ó μ,σ) |
| L2 Reglas | wᵣ = Π µᵢⱼ — firing strength por producto | — |
| L3 Normalización | w̄ᵣ = wᵣ / Σwᵣ — pesos normalizados | — |
| L4 Consecuente | fᵣ = p₁x₁ + p₂x₂ + p₃x₃ + p₀ (Sugeno lineal) | Consecuente |
| L5 Defuzzificación | ŷ = Σ w̄ᵣ·fᵣ — salida ponderada | — |

## Variables lingüísticas (3 entradas)

| Feature | Significado difuso | Conjuntos |
|---------|-------------------|----------|
| `distancia_km` | ¿Qué tan lejos está el cliente? | {Corta, Media, Larga} |
| `dias_limite_envio` | ¿Con qué urgencia debe enviarlo el vendedor? | {Urgente, Normal, Holgado} |
| `flete_total` | ¿Qué nivel de servicio se pagó? | {Bajo, Estándar, Premium} |

## Protocolo experimental (12 configuraciones)

`{2, 3, 4}` MFs × `{trimf, gaussmf}` × `{hybrid, gradient}` = **12 configuraciones**

| Parámetro | Hybrid | Gradient |
|-----------|--------|----------|
| Premisa | Init con percentiles equiespaciados → Adam | Random → Adam |
| Consecuente | Init con Mínimos Cuadrados sobre firing strengths → Adam | Random → Adam |

In [ ]:
import os

# ── CPU fallback — DEBE estar ANTES de importar TensorFlow ────────────────
os.environ['CUDA_VISIBLE_DEVICES'] = '-1'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import time
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from itertools import product as it_product

import tensorflow as tf
from tensorflow import keras
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print('Modo CPU (GPU deshabilitada)')
print(f'TensorFlow {tf.__version__}')

DATA_DIR     = Path('../dataset')
OUTPUTS_DIR  = Path('../outputs')
GRAFICAS_DIR = Path('../outputs/graficas')
MODELS_DIR   = Path('../models')
for d in [OUTPUTS_DIR, GRAFICAS_DIR, MODELS_DIR]:
    d.mkdir(parents=True, exist_ok=True)


## 1. Carga y preparación de datos

Pipeline idéntico al notebook 08 para garantizar comparabilidad.
Al final se extraen únicamente las **3 features del sistema difuso** y se normalizan a [0, 1] con MinMaxScaler (requisito del sistema difuso).

In [ ]:
# ── Cargar CSVs (solo los necesarios para las 3 features ANFIS) ───────────
orders    = pd.read_csv(DATA_DIR / 'olist_orders_dataset.csv', parse_dates=[
    'order_purchase_timestamp', 'order_delivered_customer_date',
    'order_estimated_delivery_date'])
items     = pd.read_csv(DATA_DIR / 'olist_order_items_dataset.csv', parse_dates=['shipping_limit_date'])
customers = pd.read_csv(DATA_DIR / 'olist_customers_dataset.csv')
sellers   = pd.read_csv(DATA_DIR / 'olist_sellers_dataset.csv')
geo       = pd.read_csv(DATA_DIR / 'olist_geolocation_dataset.csv')

# ── Target ─────────────────────────────────────────────────────────────────
df = orders[(orders['order_status'] == 'delivered') &
             orders['order_delivered_customer_date'].notna()].copy()
df['dias_entrega'] = (df['order_delivered_customer_date'] - df['order_purchase_timestamp']).dt.days
df = df[(df['dias_entrega'] >= 0) & (df['dias_entrega'] <= 60)]

# ── Flete y dias_limite_envio ──────────────────────────────────────────────
items_agg = items.sort_values('price', ascending=False).groupby('order_id').agg(
    flete_total=('freight_value', 'sum'),
    seller_id=('seller_id', 'first'),
    shipping_limit_date=('shipping_limit_date', 'first')
).reset_index()
df = df.merge(items_agg, on='order_id', how='left')
df['dias_limite_envio'] = (df['shipping_limit_date'] - df['order_purchase_timestamp']).dt.days.clip(lower=0)

# ── Geolocalización y distancia_km ─────────────────────────────────────────
df = df.merge(sellers[['seller_id', 'seller_zip_code_prefix', 'seller_state']],
              on='seller_id', how='left')
df = df.merge(customers[['customer_id', 'customer_zip_code_prefix', 'customer_state']],
              on='customer_id', how='left')

geo_agg = geo.groupby('geolocation_zip_code_prefix').agg(
    lat=('geolocation_lat', 'median'),
    lng=('geolocation_lng', 'median')).reset_index()

df = df.merge(geo_agg.rename(columns={'geolocation_zip_code_prefix': 'seller_zip_code_prefix',
    'lat': 'seller_lat', 'lng': 'seller_lng'}), on='seller_zip_code_prefix', how='left')
df = df.merge(geo_agg.rename(columns={'geolocation_zip_code_prefix': 'customer_zip_code_prefix',
    'lat': 'customer_lat', 'lng': 'customer_lng'}), on='customer_zip_code_prefix', how='left')

def haversine(lat1, lng1, lat2, lng2):
    R = 6371.0
    lat1, lng1, lat2, lng2 = map(np.radians, [lat1, lng1, lat2, lng2])
    a = np.sin((lat2 - lat1) / 2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin((lng2 - lng1) / 2)**2
    return 2 * R * np.arcsin(np.sqrt(a))

df['distancia_km'] = haversine(df['seller_lat'], df['seller_lng'],
                                df['customer_lat'], df['customer_lng'])

# ── Dataset final ──────────────────────────────────────────────────────────
ANFIS_FEATURES = ['distancia_km', 'dias_limite_envio', 'flete_total']
TARGET         = 'dias_entrega'

df_modelo = df[ANFIS_FEATURES + [TARGET, 'order_purchase_timestamp']].copy()
for col in ANFIS_FEATURES:
    df_modelo[col] = df_modelo[col].fillna(df_modelo[col].median())
df_modelo.dropna(subset=[TARGET], inplace=True)

# ── Split cronológico 80/10/10 (idéntico a nb08) ──────────────────────────
df_modelo = df_modelo.sort_values('order_purchase_timestamp').reset_index(drop=True)
n = len(df_modelo)
n_train, n_val = int(n * 0.80), int(n * 0.90)
train_df = df_modelo.iloc[:n_train].copy()
val_df   = df_modelo.iloc[n_train:n_val].copy()
test_df  = df_modelo.iloc[n_val:].copy()

# ── MinMaxScaler [0,1] — fit SOLO en train para evitar data leakage ───────
mms = MinMaxScaler()
X_train = mms.fit_transform(train_df[ANFIS_FEATURES]).astype('float32')
X_val   = mms.transform(val_df[ANFIS_FEATURES]).astype('float32')
X_test  = mms.transform(test_df[ANFIS_FEATURES]).astype('float32')

y_train = train_df[TARGET].values.astype('float32')
y_val   = val_df[TARGET].values.astype('float32')
y_test  = test_df[TARGET].values.astype('float32')

print(f'Filas totales : {n:,}')
print(f'Train : {len(train_df):,} | Val : {len(val_df):,} | Test : {len(test_df):,}')
print(f'\nFeatures ANFIS:')
for feat in ANFIS_FEATURES:
    print(f'  {feat:25s} — rango original [{train_df[feat].min():.1f}, {train_df[feat].max():.1f}]')
print(f'\nTarget  dias_entrega — media {y_train.mean():.1f} ± {y_train.std():.1f} días')


## 2. Arquitectura ANFIS (capa TensorFlow personalizada)

Implementación from-scratch como `tf.keras.layers.Layer`.
Soporta MF triangulares (trimf) y gaussianas (gaussmf), con parámetros aprendibles via backpropagation.

In [ ]:
class ANFISLayer(keras.layers.Layer):
    """
    Capa ANFIS Sugeno tipo-1 para n_inputs entradas y n_mfs funciones de
    membresía por entrada.

    Capas internas (Jang 1993):
      L1  — Fuzzificación:  mu[b, i, j] = MF_j(x_i)
      L2  — Firing strength: w[b, r]  = prod_i mu[b, i, idx_i(r)]   (AND por producto)
      L3  — Normalización:  w_bar[b, r] = w[b,r] / sum_r w[b,r]
      L4  — Consecuente:    f[b, r]   = p[r,:] @ x[b,:] + p0[r]
      L5  — Defuzzificación: y[b]     = sum_r (w_bar[b,r] * f[b,r])

    Parámetros premisa (MFs):
      - gaussmf: (mean, log_sigma)  por cada (input, mf)
      - trimf:   (a, m, b)          por cada (input, mf)
        con suavizado diferenciable: mu = relu(1 - |x-m|/max(b-m, m-a, eps))

    Parámetros consecuente:
      - coef: [n_rules, n_inputs+1]  (pesos lineales + bias por regla)
    """

    def __init__(self, n_inputs, n_mfs, mf_type='gaussian', **kwargs):
        super().__init__(**kwargs)
        self.n_inputs = n_inputs
        self.n_mfs    = n_mfs
        self.mf_type  = mf_type
        self.n_rules  = n_mfs ** n_inputs

        # Índices de las MFs por input para cada regla (rejilla completa)
        # shape: [n_rules, n_inputs]
        idx = list(it_product(range(n_mfs), repeat=n_inputs))
        self._rule_idx = tf.constant(idx, dtype=tf.int32)  # [n_rules, n_inputs]

    def build(self, input_shape):
        n_i, n_m = self.n_inputs, self.n_mfs

        if self.mf_type == 'gaussian':
            # mean en [0,1]; log_sigma inicializado para sigma ~ 0.3
            self.mf_mean = self.add_weight(
                name='mf_mean', shape=(n_i, n_m),
                initializer='uniform', trainable=True)
            self.mf_log_sigma = self.add_weight(
                name='mf_log_sigma', shape=(n_i, n_m),
                initializer=keras.initializers.Constant(np.log(0.30)),
                trainable=True)
        else:  # trimf
            # a < m < b en [0,1]; inicialización uniforme (se sobreescribe en hybrid)
            self.mf_a = self.add_weight(
                name='mf_a', shape=(n_i, n_m),
                initializer='uniform', trainable=True)
            self.mf_m = self.add_weight(
                name='mf_m', shape=(n_i, n_m),
                initializer='uniform', trainable=True)
            self.mf_b = self.add_weight(
                name='mf_b', shape=(n_i, n_m),
                initializer='uniform', trainable=True)

        # Consecuente lineal: [n_rules, n_inputs+1]
        self.coef = self.add_weight(
            name='coef', shape=(self.n_rules, n_i + 1),
            initializer='glorot_uniform', trainable=True)

        super().build(input_shape)

    def _fuzzify(self, x):
        """
        x: [batch, n_inputs]
        returns mu: [batch, n_inputs, n_mfs]
        """
        # x expandido: [batch, n_inputs, 1]
        x_exp = tf.expand_dims(x, axis=2)

        if self.mf_type == 'gaussian':
            sigma = tf.exp(self.mf_log_sigma) + 1e-6   # [n_inputs, n_mfs]
            mu = tf.exp(-0.5 * ((x_exp - self.mf_mean) / sigma) ** 2)
        else:  # trimf diferenciable
            a = self.mf_a  # [n_inputs, n_mfs]
            m = self.mf_m
            b = self.mf_b
            eps = 1e-6
            left  = (x_exp - a) / (m - a + eps)
            right = (b - x_exp) / (b - m + eps)
            mu = tf.clip_by_value(tf.minimum(left, right), 0.0, 1.0)

        return mu   # [batch, n_inputs, n_mfs]

    def call(self, x):
        # ── L1 Fuzzificación ───────────────────────────────────────────────
        mu = self._fuzzify(x)          # [batch, n_inputs, n_mfs]

        # ── L2 Firing strength por producto (AND diferenciable) ───────────
        # Para cada regla r con índices [j0, j1, ..., j_{n-1}]:
        #   w_r = prod_i mu[b, i, j_i]
        # Recogemos mu en el orden de la rejilla → [batch, n_rules, n_inputs]
        rule_idx_T = tf.transpose(self._rule_idx)  # [n_inputs, n_rules]
        mu_rules = tf.stack(
            [tf.gather(mu[:, i, :], rule_idx_T[i], axis=1)
             for i in range(self.n_inputs)], axis=2)  # [batch, n_rules, n_inputs]
        w = tf.reduce_prod(mu_rules, axis=2) + 1e-10   # [batch, n_rules]

        # ── L3 Normalización ──────────────────────────────────────────────
        w_sum  = tf.reduce_sum(w, axis=1, keepdims=True)  # [batch, 1]
        w_bar  = w / w_sum                                 # [batch, n_rules]

        # ── L4 Consecuente lineal (Sugeno) ────────────────────────────────
        # x_aug: [batch, n_inputs+1]  (añade bias=1)
        x_aug  = tf.concat([x, tf.ones((tf.shape(x)[0], 1), dtype=x.dtype)], axis=1)
        # f[b, r] = coef[r, :] @ x_aug[b, :]  → [batch, n_rules]
        f = tf.matmul(x_aug, tf.transpose(self.coef))    # [batch, n_rules]

        # ── L5 Defuzzificación ────────────────────────────────────────────
        y = tf.reduce_sum(w_bar * f, axis=1, keepdims=True)  # [batch, 1]
        return y


def build_anfis_model(n_mfs, mf_type):
    """Construye el modelo Keras completo con la capa ANFIS."""
    inp = keras.Input(shape=(len(ANFIS_FEATURES),), name='inputs')
    out = ANFISLayer(n_inputs=len(ANFIS_FEATURES), n_mfs=n_mfs,
                     mf_type=mf_type, name='anfis')(inp)
    model = keras.Model(inputs=inp, outputs=out, name=f'ANFIS_{mf_type}_mf{n_mfs}')
    return model


print('Clase ANFISLayer definida.')
print(f'  Features de entrada : {len(ANFIS_FEATURES)} ({", ".join(ANFIS_FEATURES)})')
print('  Reglas generadas por config:')
for n in [2, 3, 4]:
    print(f'    {n} MFs → {n**3:3d} reglas   ({n}³)')

## 3. Lógica de entrenamiento

**Hybrid**: inicializa los centros de las MFs con percentiles equiespaciados del training set y los consecuentes con Mínimos Cuadrados (LSQ) sobre los firing strengths. Luego ajusta todos los parámetros con Adam.

**Gradient**: inicialización aleatoria + Adam puro desde cero.

In [ ]:
def init_hybrid(model, X_tr, y_tr):
    """
    Inicialización híbrida:
      - Premisa  : centros equiespaciados (percentiles) en [0,1].
      - Consecuente: mínimos cuadrados sobre los firing-strengths * X para
        resolver W @ coef = y_tr  (paso forward-pass analítico).
    """
    layer = model.get_layer('anfis')
    n_mfs = layer.n_mfs

    # ── Inicializar premisa con percentiles equiespaciados ─────────────────
    centers = np.linspace(0.0, 1.0, n_mfs)  # [n_mfs]

    if layer.mf_type == 'gaussian':
        mean_init = np.tile(centers, (layer.n_inputs, 1)).astype('float32')
        layer.mf_mean.assign(mean_init)
        sigma_val = 1.0 / (n_mfs + 1)
        layer.mf_log_sigma.assign(
            np.full((layer.n_inputs, n_mfs), np.log(sigma_val), dtype='float32'))
    else:  # trimf
        step = 1.0 / (n_mfs - 1) if n_mfs > 1 else 0.5
        a_init = np.clip(centers - step, 0.0, 1.0)
        b_init = np.clip(centers + step, 0.0, 1.0)
        m_init = centers
        layer.mf_a.assign(np.tile(a_init, (layer.n_inputs, 1)).astype('float32'))
        layer.mf_m.assign(np.tile(m_init, (layer.n_inputs, 1)).astype('float32'))
        layer.mf_b.assign(np.tile(b_init, (layer.n_inputs, 1)).astype('float32'))

    # ── Inicializar consecuente con LSQ ────────────────────────────────────
    # Calcular firing strengths normalizados con los parámetros recién asignados
    X_tf  = tf.constant(X_tr, dtype=tf.float32)
    mu    = layer._fuzzify(X_tf).numpy()           # [N, n_inputs, n_mfs]

    rule_idx = layer._rule_idx.numpy()             # [n_rules, n_inputs]
    n_rules  = layer.n_rules

    # w[n, r] = prod_i mu[n, i, rule_idx[r, i]]
    w = np.ones((len(X_tr), n_rules), dtype='float32')
    for i in range(layer.n_inputs):
        w *= mu[:, i, rule_idx[:, i]]             # broadcast sobre samples
    w += 1e-10
    w_bar = w / w.sum(axis=1, keepdims=True)       # [N, n_rules]

    # Construir matriz de diseño Phi: [N, n_rules * (n_inputs+1)]
    # Para cada regla r: fila = w_bar[:,r] * [x_1, x_2, x_3, 1]
    n_inp1  = layer.n_inputs + 1
    X_aug   = np.concatenate([X_tr, np.ones((len(X_tr), 1))], axis=1)  # [N, n_inp+1]
    Phi     = np.zeros((len(X_tr), n_rules * n_inp1), dtype='float32')
    for r in range(n_rules):
        Phi[:, r * n_inp1:(r + 1) * n_inp1] = (w_bar[:, r:r+1] * X_aug)

    # Resolver Φ @ θ = y con pseudo-inversa (robusta a sistemas sobre-determinados)
    theta, _, _, _ = np.linalg.lstsq(Phi, y_tr, rcond=None)  # [n_rules * n_inp+1]
    coef_init = theta.reshape(n_rules, n_inp1).astype('float32')
    layer.coef.assign(coef_init)


def train_anfis(n_mfs, mf_type, optimizer_mode,
                epochs=200, patience=15, batch_size=512):
    """
    Entrena una configuración ANFIS y retorna (model, historia, métricas, tiempo_s, n_epocas).

    optimizer_mode: 'hybrid'   — init percentiles+LSQ, luego Adam
                    'gradient' — init aleatorio, Adam puro
    """
    tf.keras.backend.clear_session()
    tf.random.set_seed(SEED)
    np.random.seed(SEED)

    model = build_anfis_model(n_mfs, mf_type)
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3), loss='mae',
                  metrics=['mae'])

    # Inicialización
    if optimizer_mode == 'hybrid':
        init_hybrid(model, X_train, y_train)

    callbacks = [
        keras.callbacks.EarlyStopping(
            monitor='val_mae', patience=patience,
            restore_best_weights=True, mode='min', verbose=0),
        keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss', factor=0.5, patience=7,
            min_lr=1e-6, verbose=0)
    ]

    t0  = time.time()
    hist = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=epochs, batch_size=batch_size,
        callbacks=callbacks, verbose=0)
    elapsed = time.time() - t0

    # Métricas en test
    preds  = model.predict(X_test, verbose=0).flatten()
    mae    = float(mean_absolute_error(y_test, preds))
    rmse   = float(np.sqrt(mean_squared_error(y_test, preds)))
    r2     = float(r2_score(y_test, preds))
    n_ep   = len(hist.history['mae'])
    n_rules = n_mfs ** len(ANFIS_FEATURES)

    return model, hist, {'mae': mae, 'rmse': rmse, 'r2': r2,
                         'n_rules': n_rules, 'epochs': n_ep,
                         'time_s': round(elapsed, 1)}, elapsed, n_ep


print('Funciones de entrenamiento definidas.')
print(f'  Epochs máx   : 200')
print(f'  Early stopping: patience = 15 sobre val_mae')
print(f'  Batch size   : 512')
print(f'  Optimizer    : Adam lr=1e-3')

## 4. Grid Search — 12 configuraciones

`{2, 3, 4}` MFs × `{trimf, gaussmf}` × `{hybrid, gradient}` = **12 experimentos**

In [ ]:
N_MFS_LIST  = [2, 3, 4]
MF_TYPES    = ['trimf', 'gaussmf']
OPT_MODES   = ['hybrid', 'gradient']

resultados  = []     # lista de dicts con todas las métricas
modelos     = {}     # {config_id: model}  — guarda solo el mejor al final
historiales = {}     # {config_id: hist}

total = len(N_MFS_LIST) * len(MF_TYPES) * len(OPT_MODES)
print(f'Total de configuraciones: {total}')
print('=' * 72)
print(f"{'#':>2}  {'MFs':>4}  {'Tipo':>8}  {'Optim':>9}  {'Reglas':>6}  "
      f"{'MAE':>7}  {'RMSE':>7}  {'R²':>7}  {'Épocas':>6}  {'Tiempo(s)':>9}")
print('-' * 72)

cfg_id = 0
for n_mfs, mf_type, opt_mode in it_product(N_MFS_LIST, MF_TYPES, OPT_MODES):
    cfg_id += 1
    model, hist, metrics, elapsed, n_ep = train_anfis(n_mfs, mf_type, opt_mode)

    rec = {
        'id':       cfg_id,
        'n_mfs':    n_mfs,
        'mf_type':  mf_type,
        'opt_mode': opt_mode,
        **metrics
    }
    resultados.append(rec)
    modelos[cfg_id]     = model
    historiales[cfg_id] = hist

    print(f"{cfg_id:>2}  {n_mfs:>4}  {mf_type:>8}  {opt_mode:>9}  "
          f"{metrics['n_rules']:>6}  {metrics['mae']:>7.4f}  {metrics['rmse']:>7.4f}  "
          f"{metrics['r2']:>7.4f}  {metrics['epochs']:>6}  {elapsed:>9.1f}s")

print('=' * 72)

# Mejor configuración (menor MAE en test)
mejor = min(resultados, key=lambda r: r['mae'])
print(f"\n★ Mejor configuración: Config {mejor['id']} — "
      f"{mejor['n_mfs']} MFs | {mejor['mf_type']} | {mejor['opt_mode']}")
print(f"  MAE={mejor['mae']:.4f}  RMSE={mejor['rmse']:.4f}  R²={mejor['r2']:.4f}")

## 5. Análisis de resultados del grid

Gráficas comparativas: efecto de cada factor (n_mfs, tipo MF, método de optimización) sobre MAE.

In [ ]:
df_res = pd.DataFrame(resultados)

# ── Tabla completa ──────────────────────────────────────────────────────────
print('Tabla completa de resultados:')
tabla = df_res[['id', 'n_mfs', 'mf_type', 'opt_mode', 'n_rules',
                'mae', 'rmse', 'r2', 'epochs', 'time_s']].copy()
tabla.columns = ['ID', 'MFs', 'Tipo', 'Optim', 'Reglas', 'MAE', 'RMSE', 'R²', 'Épocas', 'Tiempo(s)']
tabla = tabla.sort_values('MAE')
print(tabla.to_string(index=False))

# ── Figura: heatmap MAE + barplot por factor ───────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Grid Search ANFIS — Efecto de cada factor sobre MAE (test)', fontsize=13)

# Panel 1: MAE por MFs y optimizador (promedio sobre tipos MF)
pivot1 = df_res.groupby(['n_mfs', 'opt_mode'])['mae'].mean().unstack()
im1 = axes[0].imshow(pivot1.values, cmap='RdYlGn_r', aspect='auto')
axes[0].set_xticks(range(len(pivot1.columns)))
axes[0].set_xticklabels(pivot1.columns)
axes[0].set_yticks(range(len(pivot1.index)))
axes[0].set_yticklabels([f'{v} MFs' for v in pivot1.index])
axes[0].set_title('# MFs × Optimizador\n(media sobre tipos MF)')
for i in range(len(pivot1.index)):
    for j in range(len(pivot1.columns)):
        axes[0].text(j, i, f'{pivot1.values[i, j]:.3f}',
                    ha='center', va='center', fontsize=10, fontweight='bold')
plt.colorbar(im1, ax=axes[0], label='MAE (días)')

# Panel 2: MAE medio por tipo de MF
mae_by_type = df_res.groupby('mf_type')['mae'].agg(['mean', 'min', 'max'])
colors_t = ['#4C72B0', '#DD8452']
bars = axes[1].bar(mae_by_type.index, mae_by_type['mean'],
                   color=colors_t, alpha=0.85, width=0.5)
axes[1].errorbar(range(len(mae_by_type)),
                 mae_by_type['mean'],
                 yerr=[mae_by_type['mean'] - mae_by_type['min'],
                       mae_by_type['max'] - mae_by_type['mean']],
                 fmt='none', color='black', capsize=5)
axes[1].set_title('MAE por tipo de MF\n(media ± rango)')
axes[1].set_ylabel('MAE (días)')
for bar in bars:
    axes[1].text(bar.get_x() + bar.get_width() / 2,
                 bar.get_height() + 0.02,
                 f'{bar.get_height():.3f}', ha='center', fontsize=9)

# Panel 3: MAE por modo de optimización
mae_by_opt = df_res.groupby('opt_mode')['mae'].agg(['mean', 'min', 'max'])
colors_o = ['#55A868', '#C44E52']
bars2 = axes[2].bar(mae_by_opt.index, mae_by_opt['mean'],
                    color=colors_o, alpha=0.85, width=0.5)
axes[2].errorbar(range(len(mae_by_opt)),
                 mae_by_opt['mean'],
                 yerr=[mae_by_opt['mean'] - mae_by_opt['min'],
                       mae_by_opt['max'] - mae_by_opt['mean']],
                 fmt='none', color='black', capsize=5)
axes[2].set_title('MAE por modo de optimización\n(media ± rango)')
axes[2].set_ylabel('MAE (días)')
for bar in bars2:
    axes[2].text(bar.get_x() + bar.get_width() / 2,
                 bar.get_height() + 0.02,
                 f'{bar.get_height():.3f}', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig(GRAFICAS_DIR / 'anfis_grid_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('Gráfico guardado en outputs/graficas/anfis_grid_analysis.png')

## 6. Análisis de la mejor configuración

### 6.1 Funciones de membresía aprendidas

Visualiza cómo el ANFIS adaptó los parámetros de cada variable lingüística durante el entrenamiento.

In [ ]:
mejor_model  = modelos[mejor['id']]
mejor_layer  = mejor_model.get_layer('anfis')
mejor_n_mfs  = mejor['n_mfs']
mejor_type   = mejor['mf_type']

# Etiquetas lingüísticas por número de MFs
LABELS = {
    2: ['Bajo', 'Alto'],
    3: ['Bajo', 'Medio', 'Alto'],
    4: ['Muy Bajo', 'Bajo', 'Alto', 'Muy Alto'],
}
FEATURE_LABELS = {
    'distancia_km':      {'title': 'Distancia (km)', 'unit': 'km',
                          'sets': {2: ['Cercano', 'Lejano'],
                                   3: ['Cercano', 'Medio', 'Lejano'],
                                   4: ['Muy Cercano', 'Cercano', 'Lejano', 'Muy Lejano']}},
    'dias_limite_envio': {'title': 'Plazo de envío (días)',  'unit': 'días',
                          'sets': {2: ['Urgente', 'Holgado'],
                                   3: ['Urgente', 'Normal', 'Holgado'],
                                   4: ['Muy Urgente', 'Urgente', 'Holgado', 'Muy Holgado']}},
    'flete_total':       {'title': 'Flete total (R$)', 'unit': 'R$',
                          'sets': {2: ['Económico', 'Premium'],
                                   3: ['Económico', 'Estándar', 'Premium'],
                                   4: ['Muy Econ.', 'Económico', 'Premium', 'Muy Premium']}},
}

x_range = np.linspace(0.0, 1.0, 300, dtype='float32')
colors_mf = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']

fig, axes = plt.subplots(1, len(ANFIS_FEATURES), figsize=(14, 4))
fig.suptitle(
    f'Funciones de Membresía Aprendidas — Mejor ANFIS '
    f'({mejor_n_mfs} MFs, {mejor_type}, {mejor["opt_mode"]})',
    fontsize=12, fontweight='bold')

for fi, feat in enumerate(ANFIS_FEATURES):
    ax = axes[fi]
    info = FEATURE_LABELS[feat]
    labels = info['sets'][mejor_n_mfs]

    for j in range(mejor_n_mfs):
        if mejor_type == 'gaussian':
            mean_j  = float(mejor_layer.mf_mean.numpy()[fi, j])
            sigma_j = float(np.exp(mejor_layer.mf_log_sigma.numpy()[fi, j]))
            mu_j    = np.exp(-0.5 * ((x_range - mean_j) / (sigma_j + 1e-6)) ** 2)
        else:  # trimf
            a_j = float(mejor_layer.mf_a.numpy()[fi, j])
            m_j = float(mejor_layer.mf_m.numpy()[fi, j])
            b_j = float(mejor_layer.mf_b.numpy()[fi, j])
            left  = (x_range - a_j) / (m_j - a_j + 1e-6)
            right = (b_j - x_range) / (b_j - m_j + 1e-6)
            mu_j  = np.clip(np.minimum(left, right), 0.0, 1.0)

        ax.plot(x_range, mu_j, color=colors_mf[j], linewidth=2, label=labels[j])
        ax.fill_between(x_range, mu_j, alpha=0.08, color=colors_mf[j])

    ax.set_title(info['title'], fontsize=10)
    ax.set_xlabel('Valor normalizado [0, 1]')
    ax.set_ylabel('Grado de membresía μ')
    ax.set_ylim(-0.05, 1.15)
    ax.set_xlim(0.0, 1.0)
    ax.legend(fontsize=8, loc='upper right')
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUTS_DIR / 'mfs_mejor_anfis.png', dpi=150, bbox_inches='tight')
plt.show()
print('Gráfico guardado en outputs/mfs_mejor_anfis.png')

### 6.2 Reglas lingüísticas del mejor modelo

Extracción de las reglas difusas en lenguaje natural: para cada regla activa, el consecuente lineal se convierte en un ETA estimado promedio.

In [ ]:
def generar_reglas_linguisticas(layer, feature_names, feature_labels_dict, n_mfs):
    """
    Genera las reglas difusas en lenguaje natural.
    El ETA estimado por cada regla se calcula evaluando el consecuente lineal
    en el centroide de cada conjunto difuso (pico de la gaussiana o ápice de la trimf).
    """
    rule_idx = layer._rule_idx.numpy()  # [n_rules, n_inputs]
    coef     = layer.coef.numpy()       # [n_rules, n_inputs+1]
    n_rules  = layer.n_rules
    n_inp    = layer.n_inputs

    # Centroide de cada MF para cada input
    centroids = np.zeros((n_inp, n_mfs))
    for fi in range(n_inp):
        for j in range(n_mfs):
            if layer.mf_type == 'gaussian':
                centroids[fi, j] = float(layer.mf_mean.numpy()[fi, j])
            else:  # trimf: el centroide es el ápice m
                centroids[fi, j] = float(layer.mf_m.numpy()[fi, j])

    reglas = []
    for r in range(n_rules):
        parts = []
        centroid_vals = []
        for i, feat in enumerate(feature_names):
            j     = rule_idx[r, i]
            label = feature_labels_dict[feat]['sets'][n_mfs][j]
            centroid_vals.append(centroids[i, j])
            parts.append(f'{feat.replace("_", " ")} ES {label}')
        antecedente = ' Y '.join(parts)

        # Consecuente evaluado en los centroides → ETA estimado
        x_cent  = np.array(centroid_vals + [1.0], dtype='float32')  # [n_inp+1]
        eta_est = float(np.dot(coef[r], x_cent))

        reglas.append({
            'regla': f'R{r+1:03d}: SI {antecedente}  →  ETA ≈ {eta_est:.1f} días',
            'eta': eta_est,
            'r': r
        })

    return reglas


reglas = generar_reglas_linguisticas(
    mejor_layer, ANFIS_FEATURES, FEATURE_LABELS, mejor_n_mfs)

reglas_sorted = sorted(reglas, key=lambda x: x['eta'])

print(f'Total de reglas: {len(reglas_sorted)}')
print()
print('── Reglas con MENOR ETA estimado (entregas más rápidas) ──')
for r in reglas_sorted[:5]:
    print(f"  {r['regla']}")
print()
print('── Reglas con MAYOR ETA estimado (entregas más lentas) ──')
for r in reglas_sorted[-5:]:
    print(f"  {r['regla']}")
print()
print('── Todas las reglas (por ETA) ──')
for r in reglas_sorted:
    print(f"  {r['regla']}")

# Guardar reglas en archivo de texto
reglas_path = OUTPUTS_DIR / 'reglas_mejor_anfis.txt'
with open(reglas_path, 'w', encoding='utf-8') as f:
    f.write(f'REGLAS DIFUSAS — Mejor ANFIS ({mejor_n_mfs} MFs, {mejor_type}, {mejor["opt_mode"]})\n')
    f.write(f'MAE={mejor["mae"]:.4f}  RMSE={mejor["rmse"]:.4f}  R²={mejor["r2"]:.4f}\n')
    f.write('=' * 80 + '\n\n')
    for r in reglas_sorted:
        f.write(r['regla'] + '\n')
print(f'\nReglas guardadas en {reglas_path}')


### 6.3 Curva de aprendizaje del mejor modelo

In [ ]:
hist_mejor = historiales[mejor['id']]

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(hist_mejor.history['mae'],     label='Train MAE', color='steelblue', linewidth=1.5)
ax.plot(hist_mejor.history['val_mae'], label='Val MAE',   color='darkorange',
        linestyle='--', linewidth=1.5)
best_ep = int(np.argmin(hist_mejor.history['val_mae']))
ax.axvline(best_ep, color='gray', linestyle=':', linewidth=1,
           label=f'Mejor época ({best_ep + 1})')
ax.axhline(mejor['mae'], color='red', linestyle=':', linewidth=1,
           label=f'MAE test = {mejor["mae"]:.4f}', alpha=0.7)
ax.set_title(
    f'Curva de aprendizaje — Mejor ANFIS '
    f'({mejor_n_mfs} MFs, {mejor_type}, {mejor["opt_mode"]})',
    fontsize=11)
ax.set_xlabel('Época')
ax.set_ylabel('MAE (días)')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUTS_DIR / 'curvas_anfis.png', dpi=150, bbox_inches='tight')
plt.show()
print('Curvas guardadas en outputs/curvas_anfis.png')

## 7. Comparativa final con entregas anteriores

Comparación directa de las tres entregas del proyecto usando el mismo test set y las mismas métricas.

In [ ]:
# ── Cargar métricas de entregas anteriores ────────────────────────────────
with open(OUTPUTS_DIR / 'metricas_final.json', 'r') as f:
    bl = json.load(f)
with open(OUTPUTS_DIR / 'metricas_ME.json', 'r') as f:
    me = json.load(f)

mae_bl   = bl['regresion']['mae_dias']
rmse_bl  = bl['regresion']['rmse_dias']
r2_bl    = bl['regresion']['r2']

mae_me   = me['regresion_ME']['mae']
rmse_me  = me['regresion_ME']['rmse']
r2_me    = me['regresion_ME']['r2']
feats_me = me['regresion_ME']['n_features']

mae_af  = mejor['mae']
rmse_af = mejor['rmse']
r2_af   = mejor['r2']

# ── Tabla comparativa ─────────────────────────────────────────────────────
print('╔══════════════════════════════════════════════════════════════════════╗')
print('║              COMPARATIVA FINAL — 3 ENTREGAS                         ║')
print('╠══════════════════════════════════════════════════════════════════════╣')
print(f"║  {'Modelo':<28} {'Feat':>5} {'MAE':>8} {'RMSE':>8} {'R²':>8}  ║")
print('╠══════════════════════════════════════════════════════════════════════╣')

def delta_str(new, ref, better='lower'):
    d = (new - ref) / abs(ref + 1e-8) * 100
    sign = '+' if d > 0 else ''
    arrow = '▼' if (better == 'lower' and d < 0) or (better == 'higher' and d > 0) else '▲'
    return f'({sign}{d:.1f}%{arrow})'

print(f"║  {'E1 — RN Baseline (nb08)':<28} {'15':>5} {mae_bl:>8.4f} {rmse_bl:>8.4f} {r2_bl:>8.4f}  ║")
print(f"║  {'E2 — GA-REG (nb10)':<28} {feats_me:>5} {mae_me:>8.4f} {rmse_me:>8.4f} {r2_me:>8.4f}  ║")
print(f"║    Δ vs E1:                      "
      f"      {delta_str(mae_me, mae_bl,'lower'):>12} {delta_str(rmse_me,rmse_bl,'lower'):>12} {delta_str(r2_me,r2_bl,'higher'):>12}  ║")
print('╠══════════════════════════════════════════════════════════════════════╣')
print(f"║  {'E3 — ANFIS mejor (nb11)':<28} {'3':>5} {mae_af:>8.4f} {rmse_af:>8.4f} {r2_af:>8.4f}  ║")
print(f"║    {mejor_n_mfs} MFs | {mejor_type} | {mejor['opt_mode']} | {mejor['n_rules']} reglas"
      f"{'':>16}                         ║")
print(f"║    Δ vs E1:                      "
      f"      {delta_str(mae_af, mae_bl,'lower'):>12} {delta_str(rmse_af,rmse_bl,'lower'):>12} {delta_str(r2_af,r2_bl,'higher'):>12}  ║")
print('╚══════════════════════════════════════════════════════════════════════╝')

# ── Gráfico de barras triple ──────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(13, 5))
fig.suptitle('Comparativa Final — RN Baseline vs GA-REG vs ANFIS', fontsize=13, fontweight='bold')

modelos_names = ['E1 — RN\nBaseline\n(15 feat)', f'E2 — GA\nREG\n({feats_me} feat)',
                 f'E3 — ANFIS\nMejor\n(3 feat)']
colors_bar = ['steelblue', 'darkorange', '#2ca02c']
valores = {
    'MAE (días)':  [mae_bl, mae_me, mae_af],
    'RMSE (días)': [rmse_bl, rmse_me, rmse_af],
    'R²':          [r2_bl, r2_me, r2_af],
}

for ax, (metric, vals) in zip(axes, valores.items()):
    bars = ax.bar(modelos_names, vals, color=colors_bar, alpha=0.85, width=0.5)
    ax.set_title(metric, fontsize=11)
    ax.set_ylabel(metric)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + max(vals) * 0.01,
                f'{val:.4f}', ha='center', va='bottom', fontsize=8.5)
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUTS_DIR / 'comparativo_ANFIS.png', dpi=150, bbox_inches='tight')
plt.show()
print('Comparativo guardado en outputs/comparativo_ANFIS.png')


## 8. Guardar métricas finales

In [ ]:
metricas_ANFIS = {
    'mejor_config': {
        'id':       mejor['id'],
        'n_mfs':    mejor_n_mfs,
        'mf_type':  mejor_type,
        'opt_mode': mejor['opt_mode'],
        'n_rules':  mejor['n_rules'],
        'mae':      mae_af,
        'rmse':     rmse_af,
        'r2':       r2_af,
        'epochs':   mejor['epochs'],
        'time_s':   mejor['time_s'],
    },
    'grid_resultados': resultados,
    'anfis_features': ANFIS_FEATURES,
    'comparativa': {
        'E1_baseline': {'mae': mae_bl, 'rmse': rmse_bl, 'r2': r2_bl, 'n_features': 15},
        'E2_GA_REG':   {'mae': mae_me, 'rmse': rmse_me, 'r2': r2_me, 'n_features': feats_me},
        'E3_ANFIS':    {'mae': mae_af, 'rmse': rmse_af, 'r2': r2_af, 'n_features': 3},
    }
}

out_path = OUTPUTS_DIR / 'metricas_ANFIS.json'
with open(out_path, 'w', encoding='utf-8') as f:
    json.dump(metricas_ANFIS, f, indent=2, ensure_ascii=False)

print(f'✔ metricas_ANFIS.json guardado en {out_path}')
print()
print('──── Resumen Final ────')
print(f'  Mejor ANFIS — {mejor_n_mfs} MFs | {mejor_type} | {mejor["opt_mode"]} | {mejor["n_rules"]} reglas')
print(f'  MAE  : {mae_af:.4f} días')
print(f'  RMSE : {rmse_af:.4f} días')
print(f'  R²   : {r2_af:.4f}')
print()
print('  Δ vs E1 Baseline:')
for name, val, ref, mode in [
        ('MAE ', mae_af, mae_bl, 'lower'),
        ('RMSE', rmse_af, rmse_bl, 'lower'),
        ('R²  ', r2_af, r2_bl, 'higher')]:
    d = (val - ref) / abs(ref + 1e-8) * 100
    arrow = '▼ mejor' if (mode == 'lower' and d < 0) or (mode == 'higher' and d > 0) else '▲ peor'
    print(f'    {name}: {ref:.4f} → {val:.4f}  ({d:+.1f}% {arrow})')